# Engine and Optimizations

## Overview ##

This notebook presents a simple way to use the graph_rewrite library for optimizing query execution. We would use spannerlog to write the queries and generate their execution graphs, and a modified version of spannerlib's engine defined in this notebook to run the execution graphs, while profiling the query runtime metrics.

## Imports

In [1]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)
import numpy as np
from typing import no_type_check, Set, Sequence, Any,Optional,List,Callable,Dict,Union
import networkx as nx
import itertools
from collections import defaultdict
import sys
import os
from pathlib import Path
sys.path.append("../../spannerlib/spannerlib")

import time


from spannerlib.span import Span
from spannerlib.data_types import (
    Var, 
    FreeVar, 
    RelationDefinition, 
    Relation, 
    IEFunction,
    AGGFunction,
    IERelation, 
    Rule, 
    pretty
)
from spannerlib.ra import (
    _col_names,
    get_const,
    select,
    project,
    rename,
    union,
    intersection,
    difference,
    join,
    product,
    groupby,
    ie_map,
    merge_rows
)

from spannerlib.term_graph import graph_compose, merge_term_graphs_pair,rule_to_graph,add_relation,add_project_uniq_free_vars
from spannerlib.engine import Engine, IEFunction, AGGFunction
from spannerlib import get_magic_session,Span


from graph_rewrite import draw
from graph_rewrite import rewrite, rewrite_iter


import logging
logger = logging.getLogger(__name__)

## Utils ##

Define the engine (compute_node func) and its dependencies.

The engine is a simplified version of the Spannerlib engine, designed to operate on trees. It extends the original implementation by integrating profiling capabilities.


In [2]:
def schema_match(schema,expected,ignore_types=None):
    """checks if"""
    if len(schema) != len(expected):
        return False
    if ignore_types is None:
        ignore_types = []
    for x,y in zip(schema,expected):
        if x in ignore_types:
            continue
        if not issubclass(x,y):
            return False
    return True


def is_of_schema(relation,schema,ignore_types=None):
    """checks if a relation is of a given schema"""
    try:
        if len(relation) != len(schema):
            return False
        if ignore_types is None:
            ignore_types = []
        for x,y in zip(relation,schema):
            if type(x) in ignore_types:
                continue
            if not isinstance(x,y):
                return False
        return True
    except Exception as e:
        logger.error(f"Got Error when computing:\n"
                     f"is_of_scehma({relation},{schema})\n"
                     f"Error: {e}")
        raise e

def type_merge(type1,type2):
    if issubclass(type1,type2):
        return type1
    elif issubclass(type2,type1):
        return type2
    else:
        raise ValueError(f"Trying to merge types {type1},{type2}, types are incompatible")

def schema_merge(schema1,schema2):
    """merges two schemas, taking the stricter type between the two for each index"""
    if len(schema1) != len(schema2):
        raise ValueError(f"Trying to merge schemas {schema1},{schema2} schemas must be of the same length")
    
    new_schema = [type_merge(x,y) for x,y in zip(schema1,schema2)]
    return new_schema

In [3]:
import re
STRING_PATTERN = re.compile(r"^[^\r\n]+$")

def isFloat(s):  
   n = '0123456789.' 
   return (all(x in n for x in s) and s.count('.') == 1)  
 
def isInt(s):  
   n = '0123456789'    
   return all(x in n for x in s) 

def _infer_relation_schema(row) -> Sequence[type]: # Inferred type list of the given relation
    """
    Guess the relation type based on the data.
    We support both the actual types (e.g. 'Span'), and their string representation ( e.g. `"[0,8)"`).

    **@raise** ValueError: if there is a cell inside `row` of an illegal type.
    """
    relation_types = []
    for cell in row:
        if not isinstance(cell, str):
            relation_types.append(type(cell))
        elif isInt(cell):
            relation_types.append(int)
        elif isFloat(cell):
            relation_types.append(float)
        elif cell in ['True', 'False']:
            relation_types.append(bool)
        else:
            relation_types.append(str)
        
    return relation_types

In [4]:
class DB(dict):
    def __repr__(self):
        key_str=', '.join(self.keys())
        return f'DB({key_str})'

In [5]:
def _col_names(length):
    # these names wont conflict with logical variables since they must always start with Uppercase letters
    return [f'col_{i}' for i in range(length)]

In [6]:
# some select theta functions

class equalConstTheta():
    def __init__(self,*pos_val_tuples):
        self.pos_val_tuples = pos_val_tuples
    def __call__(self,df):
        masks = [df.iloc[:,pos]==val for pos,val in self.pos_val_tuples]
        return pd.concat(masks,axis=1).all(axis=1)
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos}={val}' for pos,val in self.pos_val_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalConstTheta):
            return False
        return self.pos_val_tuples == other.pos_val_tuples

class equalColTheta():
    def __init__(self,*col_pos_tuples):
        self.col_pos_tuples = col_pos_tuples

    def __call__(self,df):
        masks = [df.iloc[:,pos1]==df.iloc[:,pos2] for pos1,pos2 in self.col_pos_tuples]
        return pd.concat(masks,axis=1).all(axis=1)    
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos1}=col_{pos2}' for pos1,pos2 in self.col_pos_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalColTheta):
            return False
        return self.col_pos_tuples == other.col_pos_tuples

In [7]:
def get_const(const_dict,**kwargs):
    return pd.DataFrame([const_dict])


def is_truthy(df):
    return df.shape==(1,0)

def is_falsy(df):
    return df.shape==(0,0)

## RA ops

In [8]:
def select(df,theta,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    if callable(theta):
        return df[theta(df)]
    else:
        raise ValueError(f"theta must be callable, got {theta}")

def project(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    return df[schema]
    
def rename(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    df=df.copy()
    df.columns = schema
    return df

def intersection(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='inner',on=list(df1.columns))

def difference(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.concat([df1,df2]).drop_duplicates(keep=False)


def product(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='cross')

def join(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or is_falsy(df1) or is_falsy(df2):
        return pd.DataFrame(columns=schema)

    # if one of the dataframes is truthy, return the other
    # this solves the problem of joining with a constant
    if is_truthy(df1):
        return df2
    if is_truthy(df2):
        return df1

    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    on = cols1 & cols2
    # get only logical variables
    # on = [ col for col in on if isinstance(col,str) and col[0].isupper()]
    on = list(on)
    if len(on)==0:
        return pd.merge(df1,df2,how='cross')
    else:
        return pd.merge(df1,df2,how='inner',on=on)
    
def merge_rows(*dfs):
    return pd.DataFrame(
        set.union(*[set(df.itertuples(index=False,name=None)) for df in dfs])
    )


def union(*dfs,schema,**kwargs):
    # use numpy arrays to ignore column names
    non_empty_dfs = []
    for df in dfs:
        if df is not None and not df.empty:
            non_empty_dfs.append(df)
    if len(non_empty_dfs)==0:
        return pd.DataFrame(columns=schema)
    else:
        return rename(merge_rows(*non_empty_dfs),schema)
        # This line didnt work since drop duplicates doesnt work correctly on non primitive classes such as Spans
        # return pd.DataFrame(np.concatenate(non_empty_dfs,axis=0),columns=schema).drop_duplicates(ignore_index=True)
        
def groupby(df,schema,agg,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    # rename columns to numbers so that we can aggregate the same free var to multiple places
    uniq_cols_df = rename(df,schema=[i for i in range(len(schema))])

    groupby_cols = [i for i,agg_func in enumerate(agg) if agg_func is None]
    agg_by_cols = {i:agg_func for i,agg_func in enumerate(agg) if agg_func is not None}
    # a real groupby
    if len(groupby_cols)>0:
        return rename(
            project(
                uniq_cols_df.groupby(groupby_cols).agg(agg_by_cols).reset_index(),
                schema = uniq_cols_df.columns
                ),
            schema)
    # no group by vars, so aggs contain all columns and schema simply orders them
    else:

        # this conversion magic is caused by an inconsistency between series and dataframes aggs,
        # to enable using both function and str aliases we
        # we take each column, convert to a frame
        # aggregate it and then squeeze it to a series (which has a single value)
        # then feed that to the dataframe constructor
        return rename(
            pd.DataFrame({
                col:[uniq_cols_df[col].to_frame().agg(agg_by_cols[col]).squeeze()] for col in range(len(agg_by_cols))
            }),
            schema)

In [9]:
def coerce_tuple_like(name,func,input,output):
    if isinstance(output,(tuple,list)):
        return output
    else:
        logger.debug(f"IEFunction {name} with underlying function {func}\n"
                        f"returned a value that is not a tuple/list\n"
                        f"for input {input} -> {output}\n"
                        f"coercing to tuple")
        return (output,)

def assert_ie_schema(name,func,value,expected_schema,arity,input_or_output='input'):
    if callable(expected_schema):
        expected_schema = expected_schema(arity)
    if not is_of_schema(value,expected_schema):
        raise ValueError(
            f"IEFunction {name} with underlying function {func}\n"
            f"received an {input_or_output} value {value}(schema={pretty(_infer_relation_schema(value))})\n"
            f"but expected {pretty(expected_schema)}")

def assert_iterable(name,func,input,output):
    try:
        out_iter = iter(output)
    except TypeError:
        raise ValueError(f"IEFunction {name} with underlying function {func}\n"
                f"returned a value that is not an iterable\n"
                f"for input {input} -> {output}")
        
def replace_f_with_c(input_string):
    if input_string.startswith("_F"):
        return input_string.replace("_F", "_C", 1)
    return input_string
        
def add_const(row,schema,in_arity,kwargs):

    in_schema = schema[:in_arity]
    new_row = []
    list_row = list(row)
    if 'const_dict' in kwargs:
        const_dict = kwargs['const_dict']
        list_cols = 0
        for col in in_schema:
            col = replace_f_with_c(col)
            if col in const_dict:
                new_row.append(const_dict[col])
            else:
                new_row.append(list_row[list_cols])
                list_cols += 1
        return new_row
    return list_row

def map_iter(df,name,func,schema,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """helper function returns an iterator that applies a function to each row of a dataframe
    """
    for _,in_row in df.iterrows():
        in_row = add_const(in_row,schema,in_arity,kwargs)
        assert_ie_schema(name,func,in_row,in_schema,in_arity,input_or_output='input')
        output = func(*in_row)
        assert_iterable(name,func,in_row,output)
        for out_row in output:
            out_row = coerce_tuple_like(name,func,in_row,out_row)
            out_row = list(out_row)
            assert_ie_schema(name,func,out_row,out_schema,out_arity,input_or_output='output')
            yield in_row + out_row


def ie_map(df,name,func,schema,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """given an indexed dataframe, apply an ie function to each row and return the output 
    such that each output relation is indexed by the same index as the input relation that generated it
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=_col_names(in_arity+out_arity))
    output_iter = map_iter(df,name,func,schema,in_schema,out_schema,in_arity,out_arity, **kwargs)
    total_arity = in_arity + out_arity
    return pd.DataFrame(output_iter,columns=schema)

## Engine

In [10]:

def get_rel(rel,db,**kwargs):
    # helper function to get the relation from the db for external relations
    return db[rel]

op_to_func = {
    'union':union,
    'intersection':intersection,
    'difference':difference,
    'select':select,
    'project':project,
    'rename':rename,
    'join':join,
    'ie_map':ie_map,
    'get_rel':get_rel,
    'get_const':get_const,
    'product':product,
    'groupby':groupby
}

In [11]:
def _in_cycle(g):
    return list(set(
        itertools.chain.from_iterable(nx.cycles.simple_cycles(g))
    ))

def _depends_on_cycle(g):
    in_cycle_nodes = _in_cycle(g)
    depends_on_cycle = {
        node for node in g.nodes if node in in_cycle_nodes or 
        len(set(nx.descendants(g,node)).intersection(in_cycle_nodes))>0
    }
    return depends_on_cycle

In [12]:
def calculate_dfs_rows(*dfs):
    if dfs:
        return sum([df.shape[0] for df in dfs])
    return 0

In [13]:
def profile_wrapper(op_func, profile_data, children_results, u_data):
    start = time.time()
    res = op_func(*children_results, **u_data)
    end = time.time()
    profile_data[op_func.__name__]["row_count"] += calculate_dfs_rows(*children_results)
    profile_data[op_func.__name__]["total_time"] += end - start
    return res, end - start

In [14]:
def _collect_children_and_run(G,u,results,profile_data,stack,log=False):
    children = list(G.successors(u))
    u_data = G.nodes[u]

    children_results = [results[v][-1] for v in children]
    op_func = op_to_func[u_data['op']]

    if log:
        logger.debug(f"computing node {u} with children {children} and data {u_data} , stack = {stack}")
        logger.debug(f"children results are {children_results}")
        logger.debug(f"children_data is {[G.nodes[v] for v in children]}")
    try:
        res, op_time = profile_wrapper(op_func, profile_data, children_results, u_data)
    except Exception as e:
        raise Exception(f'During excution of node {u} with args {children_results} and kwargs {u_data}'
                        f' got error {e}'
        )
    if log:
        logger.debug(f"result of node {u} is {res}")
    results[u].append(res)
    G.nodes[u]['op_time'] = op_time
    return res


In [15]:
def compute_node(G,root,ret_inter=False,log=False):

    # makes sure there is always a last value in the list for each key
    # which is None
    list_with_none_factory = lambda : [None]
    results_dict = defaultdict(list_with_none_factory)
    profile_data = defaultdict(lambda: {"row_count": 0, "total_time": 0.0})
    start_time = time.time()
    depends_on_cycle = _depends_on_cycle(G)
    not_depends_on_cycle = [u for u in G.nodes if u not in depends_on_cycle]

    # compute non cyclic nodes in postorder
    topological_sort = list(nx.topological_sort(nx
                                                          .DiGraph(nx.subgraph(G,G.nodes))))
    for u in topological_sort[::-1]:
        res = _collect_children_and_run(G,u,results_dict,profile_data,[],log=log)
    

    end_time = time.time()
    profile_data['total_time'] = end_time - start_time
    if ret_inter:
        return res,profile_data, results_dict
    else:
        return res, profile_data


## Queries

The query demonstrated in this notebook runs on a database of SMS messages, aiming to identify spam messages. The primary focus is on spam that falsely claims the recipient is eligible for a tax refund. The query is divided into five subqueries, each targeting a specific aspect of spam detection. All subqueries leverage regular expression-based (regex) functions for pattern matching.

I. **Year-Related Spam**  
   Detects messages containing phrases such as "refund for <year_number>" or "<year_number> open for refund," indicating potential fraud involving tax refunds for a specific year.

II. **Link-Related Spam**  
   Identifies messages with links containing keywords like "tax" or "refund," often used in spam links to deceive recipients.

III. **Money-Amount Related Spam**  
   Flags messages containing numeric patterns that resemble monetary amounts, frequently used in spam to lure victims.

IV. **Wrong Receiver**  
   Captures messages that include a "nickname" from a predefined list of known nicknames (e.g., sourced from applications like Truecaller), suggesting the message may be mistakenly or fraudulently addressed.

V. **Suspicious Sender**  
   Identifies messages sent from unknown or suspicious numbers, which are often associated with spam campaigns.


In [16]:
from spannerlib.ie_func.basic import rgx

In [17]:
magic_session = get_magic_session()

In [18]:

def rgx_is_match(pattern, # the delimeter pattern to split on
    text, # the text to be split, can be either string or Span
    ):
    """
    An IE function which given a delimeter rgx pattern and a text, 
    returns True if any match is found, False otherwise.
    """
    for _ in rgx(pattern,text):
        return [("True", )]
    return []

magic_session.register('rgx_is_match',rgx_is_match,in_schema=[str,str],out_schema=[str])

In [19]:
magic_session.import_rel('sms_rel','sms_rel_dev.csv')

In [20]:
truecaller_receiver_names = [
        'Nadav RD 15',
        'Stav Aviram - Army',
        'Stav CS Technion',
        'Nadav army',
        'Stav Compi Rep',
        'Stav Compi',
        'Stav My Love',
        'Nadav CS Technion',
        'Stav Combinatorics Partner',
        'Nadav Combi partner',
        'Nadav CS',
        'Nadav CS Rep',
        'Nadav Hamilton',
        'Nadav Numeric Algorithms',
        'Nadav the best partner',
        'Stavi'
        ]
true_caller_rgx = ''
for name in truecaller_receiver_names:
    true_caller_rgx += name + '|'

hi_rgx = f'''(?i)\\bhi\\s({true_caller_rgx})'''
hello_rgx = f'''(?i)\\bhello\\s({true_caller_rgx})'''

In [21]:
magic_session.import_var('truecaller',true_caller_rgx)
magic_session.import_var('hi_rgx',hi_rgx)
magic_session.import_var('hello_rgx',hello_rgx)


In [22]:
%%spannerlog
# tax year
sms_tax_year(sms_id,sender_id,sms_body,sms_date)<-
    sms_rel(sms_id,sender_id,sms_body,sms_date),
    rgx_is_match('refund for\s*([1-2][0-9]{3})',sms_body)->(res).

sms_tax_year(sms_id,sender_id,sms_body,sms_date)<-
    sms_rel(sms_id,sender_id,sms_body,sms_date),
    rgx_is_match('([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund',sms_body)->(res).

# tax link
sms_tax_link(sms_id,sender_id,sms_body,sms_date)<-
sms_rel(sms_id,sender_id,sms_body,sms_date),
rgx_is_match('http.*[rR][eE][fF][uU][nN][dD].*\.com',sms_body)->(res).

sms_tax_link(sms_id,sender_id,sms_body,sms_date)<-
sms_rel(sms_id,sender_id,sms_body,sms_date),
rgx_is_match('http.*[tT][aA][xX].*\.com',sms_body)->(res).


# money ammount
sms_money(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match('\d{1,3},\d{3}', sms_body) -> (res).

sms_money(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match('\d{1,3}[kK]', sms_body) -> (res).


# wrong receiver
sms_wrong_receiver(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match($hi_rgx, sms_body) -> (res).

sms_wrong_receiver(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match($hello_rgx, sms_body) -> (res).

# suspicious sender
sms_suspicious_sender(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match('\d{3}-\d+', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match('[tT][aA][xX]', sender_id) -> (res).

sms_suspicious_sender(sms_id, sender_id, sms_body, sms_date) <-
    sms_rel(sms_id, sender_id, sms_body, sms_date),
    rgx_is_match('[rR][eE][fF][uU][nN][dD]', sender_id) -> (res).


# putting it all together
sms_tax(sms_id, sender_id, sms_body, sms_date) <-
    sms_tax_year(sms_id, sender_id, sms_body, sms_date).

sms_tax(sms_id, sender_id, sms_body, sms_date) <-
    sms_tax_link(sms_id, sender_id, sms_body, sms_date).

sms_tax(sms_id, sender_id, sms_body, sms_date) <-
    sms_money(sms_id, sender_id, sms_body, sms_date).

sms_tax(sms_id, sender_id, sms_body, sms_date) <-
    sms_wrong_receiver(sms_id, sender_id, sms_body, sms_date).

sms_tax(sms_id, sender_id, sms_body, sms_date) <-
    sms_suspicious_sender(sms_id, sender_id, sms_body, sms_date).

# Optimizations

In [23]:
from copy import deepcopy
def optimize_graph(graph,rewrites):
    graph = deepcopy(graph)
    for rewrite in rewrites:
        graph = rewrite(graph)
    return graph


In [24]:
def get_source_nodes(graph):
    source_nodes = []
    for node_id in graph.nodes:
        if graph.in_degree(node_id) == 0:
            source_nodes.append(node_id)
    return source_nodes

In [25]:
def profile_optimizations(graph,before_opts,after_opts):
    
    unoptimized_graph = optimize_graph(graph,before_opts)
    unoptimized_root = get_source_nodes(unoptimized_graph)[0]
    optimized_graph = optimize_graph(graph,after_opts)
    optimized_root = get_source_nodes(optimized_graph)[0]

    res, b_profile_data = compute_node(unoptimized_graph,unoptimized_root)
    print('unoptimized')
    print(pd.DataFrame(b_profile_data).T)

    res, a_profile_data = compute_node(optimized_graph,optimized_root)
    print('optimized')
    print(pd.DataFrame(a_profile_data).T)

# Relational Optimizations

In [26]:
graph,root = magic_session.export('?sms_tax_year(sms_id,sender_id,sms_body,sms_date)',plan_query=True)#,draw_query=True)

### Removing unnecesary project nodes

In [27]:
def remove_unnecesary_projects(graph):
    rewrite(graph,
        lhs='project_node[op="project", schema]->source[schema]', p='project_node[schema],source', rhs='project_node&source',
            condition=lambda m: m['source']['schema'] == m['project_node']['schema'],
        is_recursive=True,
        )
    return graph

optimized_graph = optimize_graph(graph, [
    remove_unnecesary_projects
    ])

#draw(optimized_graph)

In [28]:
profile_optimizations(graph, [], [remove_unnecesary_projects])

unoptimized
              row_count  total_time
get_rel        0.000000    0.000002
rename       380.000000    0.001033
project     1104.000000    0.013779
get_const      0.000000    0.000614
product      364.000000    0.007787
ie_map       362.000000    0.025923
join         368.000000    0.004790
union          6.000000    0.000818
total_time     0.056428    0.056428
optimized
             row_count  total_time
get_rel       0.000000    0.000002
rename      380.000000    0.000770
get_const     0.000000    0.000300
project     736.000000    0.003587
product     364.000000    0.003197
ie_map      362.000000    0.025255
join        368.000000    0.003573
union         6.000000    0.000841
total_time    0.038234    0.038234


### Const propagation

In [29]:
def propagate_consts(graph):
    # propagate consts from all sources to target, until all sources are ie_funcs
    for m in rewrite_iter(graph, lhs='target[schema];target->source[const_dict,op]',
            p='target[schema]->source[op]',
            condition=lambda m: set(m['source']['op']) != {'ie_map'} ,
            is_recursive=True,
            ):
                new_const_dict = {k:v for d in m['source']['const_dict'] for k,v in d.items()}
                m['target']['const_dict'] = new_const_dict
                m['target']['schema'] = [col for col in m['target']['schema'] if col not in new_const_dict.keys()]

    # remove redundant const nodes
    rewrite(graph, lhs='const_node[op="get_const"];father->const_node',p='father')
    return graph

In [30]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects])
#draw(optimized_graph)

In [31]:
def remove_invalid_products(graph):
     rewrite(graph, lhs='product[op="product",schema];fathers->product->children',
             p= 'fathers,children',
            rhs='fathers->children',
            condition=lambda m: len(m['product']['schema'])==1)
     return graph

In [32]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects,remove_invalid_products])
#draw(optimized_graph)

In [33]:
# TODO: Make sure this works with all kinds of nodes - for example, joins might use the original fields of the schema to identify keys. We want to use asserts in our profiles to make sure this
#  did not change the results. To implement this we can use pandas and/or spannerlib (pd.equals)
def propagate_renames(graph):
     rewrite(graph, lhs='rename[op="rename",schema]->child[schema];fathers->rename',
            p='fathers,child[schema]',
            rhs='fathers->child[schema={{new_schema}}]',
            render_rhs={'new_schema': lambda m: m['rename']['schema']},
            is_recursive=True)
     return graph

In [34]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects,remove_invalid_products,propagate_renames])
#draw(optimized_graph)

In [35]:
def merge_identical_projects(graph):
     rewrite(graph, lhs='project[op="project",schema]->child,other_project[op="project",schema]->child',
             rhs='project&other_project->child',
            condition=lambda m: m['project']['schema'] == m['other_project']['schema'],
            is_recursive=True)
     return graph

In [36]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects,remove_invalid_products,propagate_renames,propagate_renames,
                                         merge_identical_projects])
#draw(optimized_graph)

In [37]:
def avoid_projecting_soon_to_be_removed_fields(graph):
     rewrite(graph, lhs='project[op="project",schema]->join[op="join",schema]->rel1[op="project",schema],join->rel2[op,schema]',
             rhs='project[op,schema]->join[op,schema={{j_schema}}]->rel1[op,schema={{r1_schema}}],join->rel2[op,schema]',
             render_rhs={'j_schema': lambda m: [field for field in m['join']['schema'] if field in m['project']['schema'] or field in m['rel2']['schema']],
                         'r1_schema': lambda m: [field for field in m['rel1']['schema'] if field in m['project']['schema'] or field in m['rel2']['schema']]},
               condition=lambda m: m['rel2']['op'] != 'project' or (str(m.mapping['rel1']) < str(m.mapping['rel2'])))
     return graph

In [38]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects,remove_invalid_products,propagate_renames,propagate_renames,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields])
draw(optimized_graph)

In [39]:
optimized_graph = optimize_graph(graph, [propagate_consts,remove_unnecesary_projects,remove_invalid_products,propagate_renames,propagate_renames,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields,
                                         remove_unnecesary_projects])
#draw(optimized_graph)

In [40]:
def reorder_join_and_union(graph):
         # reorder unions that has parents
        for match in rewrite_iter(graph, lhs='union[op="union",schema], union->j1[op="join"]->rel, union->j2[op="join"]->rel, j1->r1[schema], j2->r2[schema]; fathers->union',
             p='fathers, union[op,schema], j1[op]->rel, j2[op]->rel, r1[schema], r2[schema]',
             rhs='fathers->j1&j2->union[op,schema={{new_schema}}], j1&j2->rel, union->r1[schema], union->r2[schema]',
             render_rhs={'new_schema': lambda m: m['r1']['schema']},
             condition=lambda m: m['r1']['schema'] == m['r2']['schema'] and str(m.mapping['j1']) < str(m.mapping['j2'])):
             print(match)
        # reorder unions that has no parents
        '''rewrite(graph, lhs='union[op="union",schema], union->j1[op="join"]->rel, union->j2[op="join"]->rel, j1->r1[schema], j2->r2[schema]',
             p='union[op,schema], j1[op]->rel, j2[op]->rel, r1[schema], r2[schema]',
             rhs='j1&j2->union[op,schema={{new_schema}}], j1&j2->rel, union->r1[schema], union->r2[schema]',
             render_rhs={'new_schema': lambda m: m['r1']['schema']})
             #condition=lambda m: m['r1']['schema'] == m['r2']['schema'] and str(m.mapping['j1']) < str(m.mapping['j2']))'''
        return graph

In [41]:
large_graph, root = magic_session.export('?sms_tax(sms_id,sender_id,sms_body,sms_date)',plan_query=True)#,draw_query=True)

In [42]:
optimized_graph = optimize_graph(large_graph, [propagate_consts,propagate_renames,remove_unnecesary_projects,remove_invalid_products,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields,remove_unnecesary_projects])
draw(optimized_graph, ret_mermaid=True)


flowchart TB
6["6
op=#quot;ie_map#quot;, func=#lt;function rgx_is_match at 0xffff742a3eb0#gt;, in_arity=2, out_arity=1, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;], rule_id={0}, name=#quot;rgx_is_match#quot;, in_schema=[#lt;class #quot;str#quot;#gt;, #lt;class #quot;str#quot;#gt;], out_schema=[#lt;class #quot;str#quot;#gt;], const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}"]
9["9
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}"]
18["18
op=#quot;ie_map#quot;, func=#lt;function rgx_is_match at 0xffff742a3eb0#gt;, in_arity=2, out_arity=1, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;], rule_id={1}, name=#quot;rgx_is_match#quot;, in_schema=[#lt;class #quot;str#quot;#gt;, #lt;class #quot;str#quot;#gt;], out_schema=[#lt;class #quot;str#quot;#gt;], const_dict={#quot;_C0#quot;: #quot;([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund#quot;}"]
21["21
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={1}"

In [43]:
optimized_graph = optimize_graph(graph, [propagate_consts,propagate_renames,remove_unnecesary_projects,remove_invalid_products,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields,remove_unnecesary_projects,
                                         reorder_join_and_union])
draw(optimized_graph)

In [44]:
res, profile_data = compute_node(graph, root)
print(pd.DataFrame(profile_data).T)
res

              row_count  total_time
get_rel        0.000000    0.000003
rename       380.000000    0.001570
project     1104.000000    0.013647
get_const      0.000000    0.001095
product      364.000000    0.007876
ie_map       362.000000    0.031585
join         368.000000    0.006326
union          6.000000    0.001493
total_time     0.064909    0.064909


,sms_id,sender_id,sms_body,sms_date
0,S628905A234565,033-030453,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34"
1,S985456A789775,033-030575,Hi Nadav army The year 2024 has arrived and wi...,"13/06/2024, 12:02"
2,S909854A567898,Missim,Hello Stav Combinatorics Partner! You might al...,"22/08/2024, 14:06"
3,S456789A098543,TaxReturn,Stav Aviram - Army According to our records yo...,"22/01/2023, 11:32"
4,S789011A347627,073-3489675,Hi Stav Compi Information shows you have a tax...,"28/11/2023, 13:55"
5,S890985A456787,TaxBack,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07"


In [45]:
sms_rel_dev = pd.read_csv('sms_rel_dev.csv')
def find_leaf(graph):
    for node in graph.nodes:
        if graph.out_degree(node) == 0:
            return node
leaf = find_leaf(optimized_graph)

optimized_graph.nodes[leaf]['db'] = DB({'sms_rel': sms_rel_dev})

res, profile_data = compute_node(optimized_graph, get_source_nodes(optimized_graph)[0])
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


             row_count  total_time
get_rel       0.000000    0.000003
project     186.000000    0.002269
ie_map      360.000000    0.031647
join        366.000000    0.002741
union         6.000000    0.000780
total_time    0.038143    0.038143


,sms_id,sender_id,sms_body,sms_date
0,Hi Stav Compi Information shows you have a tax...,S789011A347627,073-3489675,"28/11/2023, 13:55"
1,S985456A789775,033-030575,Hi Nadav army The year 2024 has arrived and wi...,"13/06/2024, 12:02"
2,S909854A567898,Missim,Hello Stav Combinatorics Partner! You might al...,"22/08/2024, 14:06"
3,This is the last chance to withdraw. If you ha...,S628905A234565,033-030453,"10/09/2024, 10:34"
4,Stav Aviram - Army According to our records yo...,S456789A098543,TaxReturn,"22/01/2023, 11:32"
5,We noticed that you might be eligible for a ta...,S890985A456787,TaxBack,"28/08/2024, 10:07"


In [46]:
def max_sub_regex(const_dicts):
    patterns = ["([1-2][0-9]{3})"]
    const_dict1 = const_dicts[0]
    const_dict2 = const_dicts[1]
    for pattern in patterns:
        for k1, v1 in const_dict1.items():
            for _, v2 in const_dict2.items():
                if v1=="([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund" and v2=="refund for\s*([1-2][0-9]{3})" and pattern=="([1-2][0-9]{3})":
                    return {k1:pattern}
    return None  # If no matches are found

def optimize_with_sub_regex(graph):
    rewrite(graph,
             lhs='project;funcs[op="ie_map",const_dict]->project',
             p='funcs[op,const_dict],project',
             rhs='funcs[op,const_dict]->new_project[op="project",schema={{schema}}]->new_func[const_dict="max_sub_regex"]->project',
             render_rhs={'schema': lambda m: m['project']['schema']})
   
    for m in rewrite_iter(graph, lhs='func->_->new_func[const_dict="max_sub_regex"];funcs->_->new_func'):
        for k,v in m['func'].items():
            m['new_func'][k] = v
        m['new_func']['const_dict']=max_sub_regex(m['funcs']['const_dict'])

    return graph

In [47]:
optimized_graph = optimize_graph(graph, [propagate_consts, propagate_renames, remove_unnecesary_projects,remove_invalid_products,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields,
                                         remove_unnecesary_projects,reorder_join_and_union,optimize_with_sub_regex])
draw(optimized_graph)

In [48]:
relational_optimization = [propagate_consts, propagate_renames, remove_unnecesary_projects,remove_invalid_products,
                                         merge_identical_projects,avoid_projecting_soon_to_be_removed_fields,
                                         remove_unnecesary_projects,reorder_join_and_union]

In [49]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel_dev})
profile_optimizations(graph, [], relational_optimization)


unoptimized
              row_count  total_time
get_rel        0.000000    0.000003
rename       378.000000    0.000961
project     1098.000000    0.004963
get_const      0.000000    0.000375
product      362.000000    0.003204
ie_map       360.000000    0.026656
join         366.000000    0.002859
union          6.000000    0.000859
total_time     0.040665    0.040665
optimized
            row_count  total_time
get_rel       0.00000    0.000002
project     186.00000    0.001783
ie_map      360.00000    0.028815
join        366.00000    0.003031
union         6.00000    0.000798
total_time    0.03477    0.034770


In [50]:
profile_optimizations(graph, relational_optimization, relational_optimization + [optimize_with_sub_regex])

unoptimized
             row_count  total_time
get_rel       0.000000    0.000002
project     186.000000    0.004243
ie_map      360.000000    0.027447
join        366.000000    0.004688
union         6.000000    0.001004
total_time    0.038356    0.038356
optimized
             row_count  total_time
get_rel       0.000000    0.000002
project     217.000000    0.002174
ie_map      242.000000    0.030766
join        366.000000    0.003125
union         6.000000    0.000854
total_time    0.037333    0.037333


In [51]:
%spannerlog
rule(X)<-
    data(sms_body),
    rgx('greeting sentence',sms_body)->(greeting),
    rgx($ture_caller_miscalls,sms_body)->(trucall_invocation),
    is_sub_span(truecaller_invocation,greeting)->(True)


rule(x)<-
    data(sms_body),
    rgx('greeting sentence',sms_body)->(greeting),
    rgx($ture_caller_miscalls,greeting)->(trucall_invocation),

SyntaxError: invalid syntax (2116347585.py, line 2)

In [ ]:
graph, root = magic_session.export('?sms_tax(sms_id,sender_id,sms_body,sms_date)',plan_query=True)

In [ ]:
profile_optimizations(graph, [], relational_optimization)

: 

# Optimization I - subregex

In [ ]:
year_dict = {"_C0":'[1-2][0-9]{3}'}

rewrite(graph, lhs='project_node[op="project"];product_node[op="product"]->project_node',
        p='product_node[op],project_node[op]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}]\
            ,new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op]->new_project_node_2',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: year_dict,
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

draw(graph)

In [ ]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


NameError: name 'sms_rel' is not defined

# Subquery II - tax link

In [ ]:
graph, root = magic_session.export('?sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date)', plan_query=True, draw_query=True)

SEMANTIC ERROR:
During semantic checks for statement 
"?sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date)"
in pass <function verify_referenced_relations_and_functions at 0xffff52cb3a30> the following exception was raised:
Relation 'sms_tax_link' expected schema sms_tax_link(str,str,str,str) but got called with sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date)



ValueError: Relation 'sms_tax_link' expected schema sms_tax_link(str,str,str,str) but got called with sms_tax_link(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date)

In [ ]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000004
rename       773.000000    0.002175
project     1933.000000    0.023766
get_const      0.000000    0.001944
product      582.000000    0.009510
ie_map       580.000000    0.126548
join         649.000000    0.008815
union         69.000000    0.003609
total_time     0.180627    0.180627


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S567890A123459,SmartRefund,FALSE,Discover your tax refund potential today. Clic...,"27/01/2024, 11:30",TRUE
1,S890123A456791,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",TRUE
2,S789012A345680,072-3491746,FALSE,Check if you qualify for a tax refund for year...,"15/01/2024, 14:00",TRUE
3,S234567A890127,072-3491746,FALSE,Check if you qualify for a tax refund for 2023...,"07/02/2024, 08:25",TRUE
4,S012345A671113,054-5172640,FALSE,See how much you are owed in a tax refund! Sta...,"18/01/2024, 11:45",TRUE
5,S352109A847630,072-3491746,FALSE,Tax refunds simplified! Find out how much your...,"17/02/2024, 09:45",TRUE
6,S399289A990880,TaxAlerts,FALSE,"Nadav the best partner, our records show you a...","30/11/2024, 18:01",TRUE
7,S678901A234571,TaxEasy,FALSE,"Stav Combinatorics Partner, your refund could ...","11/02/2024, 16:20",TRUE
8,S567890A123455,TaxRefundSol,FALSE,"Hi Stav Compi, youre eligible for up to 100 ta...","12/08/2024, 14:21",TRUE
9,S552075A397447,TaxAlerts,FALSE,"Nadav, our records show you are eligible for a...","28/11/2024, 14:31",TRUE


# Subquery III - money amount

In [ ]:
graph, root = magic_session.export('?sms_money(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [ ]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

             row_count  total_time
get_rel        0.00000    0.000004
rename       698.00000    0.005403
project     1858.00000    0.021226
get_const      0.00000    0.004028
product      582.00000    0.012015
ie_map       580.00000    0.173862
join         620.00000    0.011268
union         40.00000    0.002828
total_time     0.23617    0.236170


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S476521A098767,SmartRefund,FALSE,Hello Stav Aviram - Army. Employees in the las...,"07/01/2024, 12:10",TRUE
1,S982123A547812,CoffeeShopPro,FALSE,Hi Nadav! Your favorite CoffeeShop is offering...,"29/11/2024, 11:45",FALSE
2,S890123A456792,FastRefunds,FALSE,"Stav Combinatorics Partner, dont miss out! Emp...","30/01/2024, 18:00",TRUE
3,S678901A234571,TaxEasy,FALSE,"Stav Combinatorics Partner, your refund could ...","11/02/2024, 16:20",TRUE
4,S345678A901233,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",TRUE
5,S901234A567901,EasyTaxRefund,FALSE,"Stav Compi Rep, Last chance! Verify your eligi...","03/01/2024, 09:15",TRUE
6,S345678A901232,TaxReturn,FALSE,"Stav Aviram - Army, Regarding the property you...","28/10/2023, 17:33",TRUE
7,S734689A482762,033-030542,FALSE,"Stav Compi, Want to know about your 2023 tax r...","16/04/2024, 10:27",TRUE
8,S234567A890122,033-030571,FALSE,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",TRUE
9,S789012A345676,Missim,FALSE,You might also receive such a message! Hello S...,"06/08/2024, 15:52",TRUE


# Subquery IV - wrong receiver

In [ ]:
graph, root = magic_session.export('?sms_wrong_receiver(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [ ]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

              row_count  total_time
get_rel        0.000000    0.000006
rename      1725.000000    0.012387
project     3465.000000    0.022865
get_const      0.000000    0.002021
product      873.000000    0.012773
ie_map       870.000000    0.225444
join        1155.000000    0.012019
union        285.000000    0.003754
total_time     0.296381    0.296381


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S890123A456789,TaxOffice35,FALSE,"Nadav combi partner, want to verify eligibilit...","21/05/2023, 20:55",TRUE
1,S123456A785432,yellow,FALSE,A new receipt has been received from Paz. To v...,"22/01/2023, 19:60",FALSE
2,S890123A456791,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",TRUE
3,S608014A732099,yellow,FALSE,A new receipt has been received from Paz. To v...,"17/06/2024, 20:23",FALSE
4,S352109A847630,072-3491746,FALSE,Tax refunds simplified! Find out how much your...,"17/02/2024, 09:45",TRUE
...,...,...,...,...,...,...
280,S123456A711123,TaxEasy,FALSE,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",TRUE
281,S688593A364525,Aya,TRUE,Meeting scheduled at 10:30 tomorrow. Dont forg...,"03/12/2024, 10:02",FALSE
282,S456789A012345,FastTAX,FALSE,"Based on our registration, you might be eligib...","15/02/2023, 21:46",TRUE
283,S903279A317893,Tomer,TRUE,Have you checked out that new place downtown? ...,"22/01/2023, 11:33",FALSE


# Subquery V - suspicious sender

In [ ]:
graph, root = magic_session.export('?sms_suspicious_sender(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [ ]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

             row_count  total_time
get_rel        0.00000    0.000006
rename      1181.00000    0.006311
project     3187.00000    0.028546
get_const      0.00000    0.001779
product      873.00000    0.014524
ie_map       870.00000    0.222524
join         976.00000    0.017479
union        372.00000    0.008505
total_time     0.30558    0.305580


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S890123A456789,TaxOffice35,FALSE,"Nadav combi partner, want to verify eligibilit...","21/05/2023, 20:55",TRUE
1,S476521A098767,SmartRefund,FALSE,Hello Stav Aviram - Army. Employees in the las...,"07/01/2024, 12:10",TRUE
2,S567890A123459,SmartRefund,FALSE,Discover your tax refund potential today. Clic...,"27/01/2024, 11:30",TRUE
3,S890123A456791,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",TRUE
4,S789012A345680,072-3491746,FALSE,Check if you qualify for a tax refund for year...,"15/01/2024, 14:00",TRUE
...,...,...,...,...,...,...
94,S456789A012345,FastTAX,FALSE,"Based on our registration, you might be eligib...","15/02/2023, 21:46",TRUE
95,S505746A208343,TaxAlerts,FALSE,"Stav CS Technion, our records show you are eli...","22/10/2024, 15:28",TRUE
96,S234567A890125,FastRefunds,FALSE,Your eligibility for a refund is pending. Clic...,"10/01/2024, 11:20",TRUE
97,S734689A482911,TaxOffice15,FALSE,"There might be 8,942 NIS waiting for you from ...","17/06/2024, 20:20",TRUE


# Putting it together

In [ ]:
# move the union to be before the join,join node, and merge both projects that project sms_body into a single project node
for match in rewrite_iter(graph,
        lhs='union_node[op="union",schema]->join_node[op="join",schema]->ie_node[func,schema],join_node->sms_rel',
        p='union_node[op],ie_node[func,schema],sms_rel',
        rhs='union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], sms_rel', 
        render_rhs={'ie_node_schema': lambda match: schema(match, 'ie_node'), 'join_node_schema': lambda match: schema(match, 'join_node')}):
        join_schema = match['join_node']['schema']
rewrite(graph, lhs='project_node[op="project"]->union_node[op="union"], sms_rel[op="get_rel",schema]', 
        p='project_node[op],sms_rel[op,schema],union_node[op]',
        rhs='project_node[op]->join_node[op="join",schema={{join_schema}}]->union_node[op], join_node->sms_rel[op,schema]',
        render_rhs={'join_schema': lambda match: join_schema})
rewrite(graph, lhs='x->project_node_1[op="project"]->sms_rel[op="get_rel"], y->project_node_2[op="project"]->sms_rel', 
        p='project_node_1[op],project_node_2[op],sms_rel[op],x,y',
        rhs='x->project_node_1&project_node_2[op]->sms_rel[op],y->project_node_1&project_node_2', is_recursive=True)
draw(graph)

In [ ]:
graph, root = magic_session.export('?sms_tax(sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam)', plan_query=True, draw_query=True)

In [ ]:
res, profile_data = compute_node(graph, root)
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

               row_count  total_time
get_rel         0.000000    0.000010
rename       5296.000000    0.033317
project     13017.000000    0.161951
get_const       0.000000    0.010565
product      3492.000000    0.084766
ie_map       3480.000000    0.924053
join         3998.000000    0.045185
union        1279.000000    0.037564
total_time      1.373886    1.373886


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S890123A456789,TaxOffice35,FALSE,"Nadav combi partner, want to verify eligibilit...","21/05/2023, 20:55",TRUE
1,S890123A456791,FreeTax,FALSE,Employees in the last 7 years are eligible for...,"16/01/2024, 08:30",TRUE
2,S123456A785432,yellow,FALSE,A new receipt has been received from Paz. To v...,"22/01/2023, 19:60",FALSE
3,S608014A732099,yellow,FALSE,A new receipt has been received from Paz. To v...,"17/06/2024, 20:23",FALSE
4,S352109A847630,072-3491746,FALSE,Tax refunds simplified! Find out how much your...,"17/02/2024, 09:45",TRUE
...,...,...,...,...,...,...
280,S123456A711123,TaxEasy,FALSE,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",TRUE
281,S688593A364525,Aya,TRUE,Meeting scheduled at 10:30 tomorrow. Dont forg...,"03/12/2024, 10:02",FALSE
282,S456789A012345,FastTAX,FALSE,"Based on our registration, you might be eligib...","15/02/2023, 21:46",TRUE
283,S903279A317893,Tomer,TRUE,Have you checked out that new place downtown? ...,"22/01/2023, 11:33",FALSE


# Relational Optimization

The graph created by spannerlog uses redundant/unnecessary relational logic. We start by optimizing the graph to use only the minimal necessary logic.

In [ ]:
raw_graph = graph.copy()

The schema's column names in the graph are col_0, col_1 ... instead of the schema of the input relation. 

Notice, that any rename node's schema can be propagated to its child's schema, making the rename obsolete. Thus, we make the following rewrite:


In [ ]:
rewrite(graph, lhs='y[op="rename",schema]->z[schema];x->y',
        p='x,z[schema]',
        rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=1.0013580322265625e-05"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={#quot;fact#quot;, 0, 12, 1}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0028259754180908203"]
1["1
op=#quot;project#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], rule_id={0}, op_time=0.008023738861083984"]
2["2
op=#quot;project#quot;, schema=

Many projects in the graph are actually redundant - They project the entire schema of their child's node. 

In our case, we notice that the only projects that are not redundant are projects whose parents are product nodes (Since they project only the relevant columns and not the entire schema), and projects whose child is a product node (Since those projects' parents are ie_map nodes, and the project defines the order of the input columns to the ie function)

In [ ]:
def is_not_product(match, node):
    if 'product' in match[node]['op']:
        return False
    return True
rewrite(graph, lhs='project_node[op="project", schema];x->project_node->y', p='x,y', rhs='x->y',condition=lambda match:
        is_not_product(match, 'x') and is_not_product(match, 'y'), is_recursive=True)


In [ ]:

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=1.0013580322265625e-05"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={#quot;fact#quot;, 0, 12, 1}, op=#quot;union#quot;, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op_time=0.0028259754180908203"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0007376670837402344"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.000

IE nodes' parents are join nodes, that join the result of the ie function and sms_rel's schema. After the join operation, there is a union node for each of the subqueries. A logical optimization is making the union before the join, s.t. there is one join operation for each subquery.

In [ ]:
def collection_schema(match, node):
    return match[node]['schema'][0]

rewrite(graph,
        lhs='union_node[op="union",schema];union_node->join_node[op="join",schema]->sms_rel[op="get_rel"],parent_union[op="union",schema]->union_node,join_node->ie_node[func,schema]',
        p='parent_union[op,schema],union_node[op],ie_node[func,schema],sms_rel[op]',
        rhs='parent_union[op,schema]->project_node[op="project",schema={{parent_union_schema}}]->join_node[op="join",schema={{join_node_schema}}]->union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], join_node->sms_rel[op]', 
        render_rhs={'ie_node_schema': lambda match: collection_schema(match, 'ie_node'), 'join_node_schema': lambda match: collection_schema(match, 'join_node'),
                    'parent_union_schema': lambda match: collection_schema(match, 'parent_union')},
        is_recursive=True)

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=1.0013580322265625e-05"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={#quot;fact#quot;, 0, 12, 1}, op=#quot;union#quot;, op_time=0.0028259754180908203, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
2["2
op=#quot;project#quot;, schema=[#quot;sms_body#quot;], rule_id={0}, op_time=0.0007376670837402344"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00041222572326660156"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C

The final relational optimization involves merging identical project nodes. In the current execution graph, each subquery contains multiple project nodes that project the same single column from the `sms_rel` schema. Specifically, four subqueries project the `sms_body` column, while the fifth subquery projects the `sender_id` column. By merging these nodes, each column is projected only once, reducing redundant operations and improving efficiency.








In [ ]:

rewrite(graph, lhs='project_node[op="project",schema]->sms_rel[op="get_rel"]',
        p='sms_rel[op]',
        is_recursive=True)

rewrite(graph, lhs='sms_rel[op="get_rel"]',
        rhs='sms_rel[op],sms_body_project_node[op="project",schema={{sms_body}},sms_body]->sms_rel[op],\
                sender_id_project_node[op="project",schema={{sender_id}},sender_id]->sms_rel[op]',
        render_rhs={'sms_body': lambda match: ['sms_body'], 'sender_id': lambda match: ['sender_id']})   

rewrite(graph, lhs='sms_body[sms_body],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sms_body[sms_body]',
        condition=lambda match: 'sms_body' in schema(match, 'product_node'))

rewrite(graph, lhs='sender_id[sender_id],product_node[op="product",schema]',
        rhs='product_node[op,schema]->sender_id[sender_id]',
        condition=lambda match: 'sender_id' in schema(match, 'product_node'))

draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year, sms_tax_link, sms_money, sms_wrong_receiver, sms_suspicious_sender, sms_tax), op_time=1.0013580322265625e-05"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={#quot;fact#quot;, 0, 12, 1}, op=#quot;union#quot;, op_time=0.0028259754180908203, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.00041222572326660156"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.0046651363372802734"]
5["5
op=#quot;project#quot;, schema=[#quot;_C0#quo

In [ ]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
               row_count  total_time
get_rel         0.000000    0.000010
rename       5296.000000    0.033317
project     13017.000000    0.161951
get_const       0.000000    0.010565
product      3492.000000    0.084766
ie_map       3480.000000    0.924053
join         3998.000000    0.045185
union        1279.000000    0.037564
total_time      1.373886    1.373886
Profile data after optimization
              row_count  total_time
get_rel        0.000000    0.000005
project     4849.000000    0.021805
get_const      0.000000    0.009497
product     3480.000000    0.063423
ie_map      3468.000000    0.686964
union       1036.000000    0.017994
join        1904.000000    0.021266
total_time     0.826202    0.826202


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True
1,S982123A541122,SpaHaven,False,Hi Nadav! Relax and rejuvenate with SpaHavens ...,"30/11/2024, 09:47",False
2,S678901A234567,Odll Global Solution,False,Your 2017 tax refund is waiting! Up to 20K NIS...,"06/03/2023, 11:35",True
3,S734689A433668,VIVINO,False,"Hello Stav Aviram, Your reservation for VIVINO...","17/07/2024, 16:00",False
4,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
...,...,...,...,...,...,...
280,S821639A531892,Boaz,True,I saw the funniest memetotally reminded me of ...,"06/11/2024, 09:28",False
281,S567890A123456,TAXES,False,"Stav Combinatorics Partner, reminder for you t...","05/03/2023, 20:42",True
282,S936463A791145,Rachel,True,Thinking about youhope all is well!,"30/11/2024, 09:49",False
283,S815997A181979,M_Hashuk,False,Your order from Makhsanei Hashuk has been coll...,"02/11/2024, 20:22",False


# Subregex optimization

This optimization targets the IE functions within each subquery. Each subquery is created using multiple regex IE functions. By identifying the largest mutual subregex among these functions and filtering rows based on it, the input dataset for each IE function is reduced, resulting in faster query execution.

For each subquery, the following structure of nodes is added for the newly introduced IE function:

`rename_node`->`ie_node`->`project_node[schema=ie_node_input_schema"]`->`product_node`->`mutual_subregex_node`, `product_node`->`project_node[schema="sms_body"/"sender_id"]`

In [ ]:
relational_optimized_graph = graph.copy()

In [ ]:
def largest_subregex_rewrite(g, rel, largest_subregex):
    rewrite(g, lhs='project_node[op="project"],union_node[rel="'+rel+'"];union_node->_->_->product_node[op="product"]->project_node',
        p='product_node[op],project_node[op],union_node[rel]',
        rhs='new_ie_node[op="ie_map",func={{func}},in_arity=2,out_arity=1,schema={{ie_schema}},name="rgx_is_match",in_schema={{in_schema}},out_schema={{out_schema}}],\
            new_product_node->new_get_const_node[op="get_const",const_dict={{const_dict}},schema={{const_schema}}],\
            new_product_node[op="product",schema={{product_schema}}]->project_node[op],\
            new_project_node_2[op="project",schema={{project_schema_2}}]->new_rename_node[op="rename",schema={{ie_schema}}]->new_ie_node\
            ->new_project_node_1[op="project",schema={{project_schema_1}}]->new_product_node,\
            product_node[op,rel]->new_project_node_2,\
            union_node[rel]',
        render_rhs={'func': lambda match: rgx_is_match,
                    'ie_schema': lambda match: ['_F0', 'sms_body', 'res'],
                    'product_schema': lambda match: ['sms_body', '_C0'],
                    'const_schema': lambda match: ['_C0'],
                    'project_schema_1': lambda match: ['_C0', 'sms_body'],
                    'project_schema_2': lambda match: ['sms_body'],
                    'const_dict': lambda match: largest_subregex,
                    'in_schema': lambda match: [str,str],
                    'out_schema': lambda match: [str]})

In [ ]:
rel_to_largest_subregex = {'sms_tax_year': {"_C0":'[1-2][0-9]{3}'},
                           'sms_tax_link': {"_C0":'http.*\.com'},
                           'sms_money': {"_C0":'\d{1,3}'},
                           'sms_wrong_receiver': {"_C0":'Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|'}
}
for rel in rel_to_largest_subregex:
    largest_subregex_rewrite(graph, rel, rel_to_largest_subregex[rel])
draw(graph,ret_mermaid=True)


flowchart TB
sms_rel["sms_rel
rel=#quot;sms_rel#quot;, rule_id={0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, #quot;fact#quot;}, schema=[#quot;sms_id#quot;, #quot;sender_id#quot;, #quot;is_sender_in_contacts#quot;, #quot;sms_body#quot;, #quot;sms_date#quot;, #quot;is_spam#quot;], op=#quot;get_rel#quot;, db=DB(sms_rel, sms_tax_year), op_time=5.0067901611328125e-06"]
sms_tax_year["sms_tax_year
rel=#quot;sms_tax_year#quot;, rule_id={#quot;fact#quot;, 0, 12, 1}, op=#quot;union#quot;, op_time=0.0016298294067382812, schema=[#quot;_F0#quot;, #quot;sms_body#quot;, #quot;res#quot;]"]
3["3
op=#quot;get_const#quot;, const_dict={#quot;_C0#quot;: #quot;refund for\\s*([1-2][0-9]{3})#quot;}, schema=[#quot;_C0#quot;], rule_id={0}, op_time=0.0004298686981201172"]
4["4
op=#quot;product#quot;, schema=[#quot;sms_body#quot;, #quot;_C0#quot;], rule_id={0}, op_time=0.00656890869140625, rel=None"]
5["5
op=#quot;project#quot;, schema=[#quot;_C0#quot;, #quot;sms_body#quot;], rule_id={0}, op_time=0.001081228256225586"]

In [ ]:
graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax': pd.DataFrame()})
print("Profile data before optimization:")
before_optimization_profile_df = pd.DataFrame(profile_data).T
print(before_optimization_profile_df)
res, profile_data = compute_node(graph, root)
print("Profile data after optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

Profile data before optimization:
              row_count  total_time
get_rel        0.000000    0.000005
project     4849.000000    0.021805
get_const      0.000000    0.009497
product     3480.000000    0.063423
ie_map      3468.000000    0.686964
union       1036.000000    0.017994
join        1904.000000    0.021266
total_time     0.826202    0.826202
Profile data after optimization:
              row_count  total_time
get_rel        0.000000    0.000006
get_const      0.000000    0.011538
project     5943.000000    0.139277
product     3828.000000    0.111468
ie_map      3812.000000    1.623862
rename       750.000000    0.003128
union       1036.000000    0.027634
join        1904.000000    0.029382
total_time     1.960823    1.960823


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True
1,S982123A541122,SpaHaven,False,Hi Nadav! Relax and rejuvenate with SpaHavens ...,"30/11/2024, 09:47",False
2,S678901A234567,Odll Global Solution,False,Your 2017 tax refund is waiting! Up to 20K NIS...,"06/03/2023, 11:35",True
3,S734689A433668,VIVINO,False,"Hello Stav Aviram, Your reservation for VIVINO...","17/07/2024, 16:00",False
4,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
...,...,...,...,...,...,...
280,S821639A531892,Boaz,True,I saw the funniest memetotally reminded me of ...,"06/11/2024, 09:28",False
281,S567890A123456,TAXES,False,"Stav Combinatorics Partner, reminder for you t...","05/03/2023, 20:42",True
282,S936463A791145,Rachel,True,Thinking about youhope all is well!,"30/11/2024, 09:49",False
283,S815997A181979,M_Hashuk,False,Your order from Makhsanei Hashuk has been coll...,"02/11/2024, 20:22",False


In [ ]:
subregex_optimized_graph = relational_optimized_graph.copy()
g_rel_to_largest_subregex = {'sms_tax_year': {"_C0":'[1-2][0-9]{3}'},
                            'sms_tax_link': {"_C0":'http.*\.com'},
                            'sms_money': {"_C0":'\d{1,3}[kK,]'},
#                            'sms_wrong_receiver': {"_C0":'Nadav RD 15|Stav Aviram - Army|Stav CS Technion|Nadav army|Stav Compi Rep|Stav Compi|Stav My Love|Nadav CS Technion|Stav Combinatorics Partner|Nadav Combi partner|Nadav CS|Nadav CS Rep|Nadav Hamilton|Nadav Numeric Algorithms|Nadav the best partner|Stavi|'}
}
for rel in g_rel_to_largest_subregex:
    largest_subregex_rewrite(subregex_optimized_graph, rel, rel_to_largest_subregex[rel])

In [ ]:
large_sms_rel = pd.read_csv('large_sms_rel.csv')
relational_optimized_graph.nodes['sms_rel']['db'] = DB({'sms_rel':large_sms_rel, 'sms_tax': pd.DataFrame()})
subregex_optimized_graph.nodes['sms_rel']['db'] = DB({'sms_rel': large_sms_rel, 'sms_tax': pd.DataFrame()})
raw_graph.nodes['sms_rel']['db'] = DB({'sms_rel': large_sms_rel, 'sms_tax': pd.DataFrame()})
res, profile_data = compute_node(relational_optimized_graph, root)
print("Profile data before optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res, profile_data = compute_node(subregex_optimized_graph, root)
print("Profile data after optimization:")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

Profile data before optimization:
                row_count  total_time
get_rel          0.000000    0.000005
project     171470.000000    0.020346
get_const        0.000000    0.003802
product     131760.000000    0.121489
ie_map      131748.000000   40.903648
union        18190.000000    0.196907
join         63934.000000    0.110962
total_time      41.372118   41.372118
Profile data after optimization:
                row_count  total_time
get_rel          0.000000    0.000004
get_const        0.000000    0.014843
project     150371.000000    0.040050
product     106718.000000    0.090909
ie_map      106703.000000   27.019848
rename        3946.000000    0.000735
union        18190.000000    0.135527
join         63934.000000    0.039728
total_time      27.347634   27.347634


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S789011G347648,Yaron,True,I'm always drunk on my own tears ((:,"29/11/2021, 18:13",False
1,S359827D909853,Adir,True,thinking your future was me ((:,"07/01/2023, 21:41",False
2,S911854E567929,Danielle,True,And when you are young (-;,"15/08/2021, 15:58",False
3,S511890D985454,Omer,True,your cheeks were turning red :)),"06/09/2022, 19:16",False
4,S098545F678939,Harel,True,I wanna be your end game ;-),"17/08/2024, 23:39",False
...,...,...,...,...,...,...
8664,S456789D058165,Yiftach,True,And no one here's to blame (-;,"18/05/2021, 22:46",False
8665,S511890F985454,Ziv,True,I was hittin' my marks :D,"17/11/2021, 20:57",False
8666,S456789G098554,Emanuel,True,Who's afraid of little old me? ;-),"21/01/2023, 18:44",False
8667,S359827D909906,Rotem,True,Run into you sometimes :),"21/06/2023, 10:14",False


In [ ]:
total_time_before = 0
total_time_after_relational = 0
total_time_after_subregex = 0
for _ in range(5):
    res, profile_data = compute_node(raw_graph, root)
    total_time_before += profile_data['total_time']
    res, profile_data = compute_node(relational_optimized_graph, root)
    total_time_after_relational += profile_data['total_time']
    res, profile_data = compute_node(subregex_optimized_graph, root)
    total_time_after_subregex += profile_data['total_time']
print("Total time before optimization:", total_time_before)
print("Total time after relational optimization:", total_time_after_relational)
print("Total time after subregex optimization:", total_time_after_subregex)
print("Total time saved by relational optimization:", total_time_before - total_time_after_relational)
print("Total time saved by subregex optimization:", total_time_after_relational - total_time_after_subregex)
print("Total time saved:", total_time_before - total_time_after_subregex)
print("Average time relational optimization saved:", (total_time_before - total_time_after_relational) / 100)
print("Average time subregex optimization saved:", (total_time_after_relational - total_time_after_subregex) / 100)
print("Average total time saved:", (total_time_before - total_time_after_subregex) / 100)

KeyboardInterrupt: 

## Subquery I - tax year ##

In [ ]:
graph,root = magic_session.export('?sms_tax_year(sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date)',plan_query=True,draw_query=True)

In [ ]:
res, profile_data = compute_node(graph,root)